# 02h — Ruler Audit and Corpus Repair

**Accelerator: NONE. This notebook must not consume GPU quota.**

**Only input required:** the notebook output containing `train.jsonl` / `val.jsonl`
(notebook 01's output). All tooling is embedded below — nothing else to attach.

## Why this notebook exists

Every `n_phonemes` label in the base training corpus is a non-space **character** count,
not a phoneme count. `phonemize_text` ended in a bare
`except Exception: return [c for c in text if not c.isspace()]`; espeak-ng is a *system*
package that `pip install phonemizer` does not install; it was absent from the
corpus-generation session, so the fallback fired for every row while logging at DEBUG.

Verified locally on the augmented subset: over 1,289 base rows, `n_phonemes` equals the
non-space character count **100.0%** of the time, with the chars/label ratio at exactly
1.000 for both the minimum and the maximum. That is an identity, not a correlation.

The length-augmentation session *did* have espeak-ng, so its 1,064 rows carry real phoneme
counts. The corpus therefore mixes two incompatible units under one prompt token,
`[Target Phonemes: N]`, and a model trained on the union is being taught two contradictory
tasks. That is the most likely reason length slope sits at 0.687 rather than near 1.0 —
and it is not a modelling problem at all.

## What this notebook decides

Whether the existing checkpoint is **salvageable by rescaling** or must be **retrained on
relabelled data**. The test is whether phonemes are a near-constant multiple of characters
*within* a language. If they are, the model learned the right capability in the wrong unit,
and a per-language constant recovers it at inference time for free. If they are not, the
character label was too weakly coupled to duration to have taught budget obedience at all.

Thresholds are pre-registered in the embedded `tools/ruler_audit.py`, written before the
numbers are seen, so the answer cannot be rationalised afterwards.


## 1. System dependencies

espeak-ng is a **system** package. `pip install phonemizer` does not install it, and that gap is exactly how the original mislabelling happened without a symptom.

In [ ]:
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y espeak-ng > /dev/null 2>&1
!pip -q install phonemizer indic-nlp-library 2>&1 | tail -2

import subprocess
print(subprocess.run(["espeak-ng", "--version"], capture_output=True, text=True).stdout.strip())

# indic-nlp-library canonicalises the script BEFORE G2P. Its basic normalisers need no
# `indic_nlp_resources` download (verified across all 11 languages) — only transliteration
# and morphology do.
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
_n = IndicNormalizerFactory().get_normalizer("hi")
assert _n.normalize("\u0958") == "\u0915\u093c", "normaliser is not canonicalising nukta forms"
print("IndicNLP normalisation: OK")


## 2. Embedded tooling

Written out verbatim from the repo so this notebook is reproducible on its own. Identical to `pipeline_v3/` — if you change one, regenerate the other, or you have reintroduced the class of drift that caused this.

In [ ]:
import os, sys
os.makedirs("/kaggle/working/pipeline_v3/common", exist_ok=True)
os.makedirs("/kaggle/working/pipeline_v3/tools", exist_ok=True)
os.chdir("/kaggle/working/pipeline_v3")
sys.path.insert(0, "/kaggle/working/pipeline_v3")
print(os.getcwd())


In [ ]:
%%writefile /kaggle/working/pipeline_v3/common/__init__.py
"""Shared utilities used across translation/, training/, and tts/: language metadata
(languages.py) and CTC forced alignment (forced_alignment.py)."""


In [ ]:
%%writefile /kaggle/working/pipeline_v3/common/languages.py
"""
common/languages.py
--------------------
Single source of truth for language metadata used across pipeline_v3.

Every other module (dataset_generator, duration_predictor, isochrony_translation_v3,
train_translation_llm, data_augmentation, train_tts) imports from here instead of
hard-coding its own language table. This avoids the classic bug where "hi" is spelled
one way in one file and another way in another file.

The 11 languages below are the intersection of:
  - Samanantar's 11 Indic targets (as, bn, gu, hi, kn, ml, mr, or, pa, ta, te)
  - IndicF5's 11 supported languages (same set)
  - espeak-ng's Indic voice coverage (verified: all 11 present as of espeak-ng 1.51)

NOTE ON EXPANSION RATIOS: The `heuristic_expansion_ratio` and `heuristic_phonemes_per_sec`
values below are *cold-start defaults*, carried forward from/consistent with the existing
V2 pipeline's phoneme_counter.py approach (e.g. Hindi=1.30, Tamil=1.35). They are
deliberately approximate. The entire point of V3 is to replace these hard-coded numbers
with the learned DurationPredictor (see translation/duration_predictor.py) once you have
trained it on real forced-aligned speech. Treat this table as the fallback that is used
(a) before you've trained anything, and (b) as a sanity-check ceiling/floor on predicted
durations even after training.
"""

from dataclasses import dataclass


@dataclass(frozen=True)
class LanguageInfo:
    name: str                          # Human-readable display name
    iso_code: str                      # 2-letter ISO 639-1 code, used by Samanantar & IndicF5
    espeak_code: str                   # espeak-ng voice code (for phonemizer backend="espeak")
    indicf5_code: str                  # Code IndicF5 expects (matches iso_code for all 11)
    script: str                        # Unicode script name, for sanity checks / logging
    heuristic_expansion_ratio: float   # Indic-text-length / English-text-length, rough prior
    heuristic_phonemes_per_sec: float  # Cold-start speaking rate for DurationPredictor fallback


# fmt: off
LANGUAGES: dict[str, LanguageInfo] = {
    "hi": LanguageInfo("Hindi",      "hi", "hi", "hi", "Devanagari", 1.30, 13.5),
    "bn": LanguageInfo("Bengali",    "bn", "bn", "bn", "Bengali",    1.28, 13.0),
    "mr": LanguageInfo("Marathi",    "mr", "mr", "mr", "Devanagari", 1.27, 13.2),
    "gu": LanguageInfo("Gujarati",   "gu", "gu", "gu", "Gujarati",   1.25, 13.0),
    "pa": LanguageInfo("Punjabi",    "pa", "pa", "pa", "Gurmukhi",   1.22, 12.8),
    "ta": LanguageInfo("Tamil",      "ta", "ta", "ta", "Tamil",      1.35, 12.5),
    "te": LanguageInfo("Telugu",     "te", "te", "te", "Telugu",     1.33, 12.6),
    "kn": LanguageInfo("Kannada",    "kn", "kn", "kn", "Kannada",    1.30, 12.7),
    "ml": LanguageInfo("Malayalam",  "ml", "ml", "ml", "Malayalam",  1.38, 12.3),
    "or": LanguageInfo("Odia",       "or", "or", "or", "Odia",       1.26, 13.0),
    "as": LanguageInfo("Assamese",   "as", "as", "as", "Bengali",    1.27, 12.9),
}
# fmt: on

# Samanantar's HF dataset config names are lowercase 2-letter codes identical to iso_code,
# EXCEPT the source side, which is always English.
SAMANANTAR_SOURCE_LANG = "en"
SAMANANTAR_HF_PATH = "ai4bharat/samanantar"

# ai4bharat/Kathbath directory structure uses full language *names* (lowercase), not codes.
KATHBATH_DIR_NAMES: dict[str, str] = {
    "hi": "hindi", "bn": "bengali", "mr": "marathi", "gu": "gujarati",
    "pa": "punjabi", "ta": "tamil", "te": "telugu", "kn": "kannada",
    "ml": "malayalam", "or": "odia", "as": "assamese",
    # Kathbath additionally has Sanskrit/Urdu/Nepali/Chhattisgarhi splits not covered here
    # because they fall outside the 11-language Samanantar/IndicF5 intersection.
}
KATHBATH_HF_PATH = "ai4bharat/Kathbath"

# ai4bharat/Rasa is the (much closer to IndicF5's own training mix) TTS-quality speech
# dataset: studio recordings, 13 languages, ~500+ hours, MIT-tagged on HF. Prefer this over
# Kathbath (which is ASR-oriented, phone/field recordings) for anything TTS-related, and
# fall back to Kathbath only if you need more raw hours than Rasa provides for a language.
RASA_HF_PATH = "ai4bharat/Rasa"

# ai4bharat/BPCC (Bharat Parallel Corpus Collection) - the successor to Samanantar:
# ~230M pairs, 22 languages, and (unlike standalone Samanantar's ambiguous HF license
# tag) an EXPLICIT license table on its dataset card: all MINED corpora - including
# Samanantar (19.4M) and Samanantar++ (121.6M) - are CC0; the human-annotated seed
# subsets (BPCC-H-Wiki/Daily) are CC-BY-4.0. This resolves the Samanantar license
# ambiguity flagged in dataset_generator.py for commercial use. GATED: accept the
# conditions on huggingface.co/datasets/ai4bharat/BPCC with your HF account first.
#
# STRUCTURE (verified 2026-07-13 by browsing the gated repo directly, including
# AI4Bharat's own compile.py in the repo root, which generated these files):
# BPCC is a RAW-FILE repo, not a config-per-language dataset - the HF dataset viewer is
# disabled for it and it has no loading script, so load_dataset("ai4bharat/BPCC",
# <config>) does NOT work. The data lives in per-subset directories of per-language
# TSVs keyed by FLORES-200 code:
#     <subset>/<flores_code>.tsv    e.g.  samanantar_v2/hin_Deva.tsv  (6.9 GB)
# Each TSV is tab-separated WITH a header row and exactly these columns (from
# compile.py: df.to_csv(sep='\t', index=False)):
#     src_lang  tgt_lang  src  tgt      (src = English text, tgt = Indic text)
# All 11 of our languages are present in samanantar_v2/ (plus npi/urd, unused here).
# dataset_generator.py streams these via load_dataset("csv", data_files="hf://...").
# Subsets seen in the repo: samanantar_v2 (default - 28.9GB, the filtered mined set),
# samanantar_v0.3_filtered, nllb_filtered, nllb_seed, bpcc-seed-latest/v1/v2,
# comparable, daily, ilci, massive, wiki. Only samanantar_v2's internal layout was
# inspected; treat other subsets' layouts as unverified until probed.
BPCC_HF_PATH = "ai4bharat/BPCC"
BPCC_SOURCE_FLORES = "eng_Latn"
BPCC_DEFAULT_SUBSET = "samanantar_v2"
BPCC_FLORES_CODES: dict[str, str] = {
    "hi": "hin_Deva", "bn": "ben_Beng", "mr": "mar_Deva", "gu": "guj_Gujr",
    "pa": "pan_Guru", "ta": "tam_Taml", "te": "tel_Telu", "kn": "kan_Knda",
    "ml": "mal_Mlym", "or": "ory_Orya", "as": "asm_Beng",
}

# IndicF5 itself
INDICF5_HF_PATH = "ai4bharat/IndicF5"
INDICF5_SAMPLE_RATE = 24000

# ai4bharat/vits_rasa_13 - fixed-inventory multi-speaker VITS TTS (40.2M params,
# CC-BY-4.0, GATED). Everything below is transcribed 1:1 from the model card's own
# speaker/style tables (verified 2026-07-13 via authenticated access, not assumed).
# IMPORTANT COVERAGE FACT: the model supports 13 languages but NOT Hindi, Gujarati, or
# Odia - three of this pipeline's 11 targets. hi/gu/or must use the IndicF5 backend
# for multi-speaker dubbing (see tts/vits_rasa_tts.py and multi_speaker_dubbing.py).
VITS_RASA_HF_PATH = "ai4bharat/vits_rasa_13"
VITS_RASA_SPEAKERS: dict[str, int] = {
    "ASM_F": 0, "ASM_M": 1, "BEN_F": 2, "BEN_M": 3, "BRX_F": 4, "BRX_M": 5,
    "DOI_F": 6, "DOI_M": 7, "KAN_F": 8, "KAN_M": 9, "MAI_M": 10, "MAL_F": 11,
    "MAR_F": 12, "MAR_M": 13, "NEP_F": 14, "PAN_F": 15, "PAN_M": 16, "SAN_M": 17,
    "TAM_F": 18, "TEL_F": 19,
}
# Style/emotion IDs exactly as published (note the real gaps at 9/11/13 - the model
# card skips those IDs; do not "fix" this by renumbering).
VITS_RASA_STYLES: dict[str, int] = {
    "ALEXA": 0, "ANGER": 1, "BB": 2, "BOOK": 3, "CONV": 4, "DIGI": 5, "DISGUST": 6,
    "FEAR": 7, "HAPPY": 8, "NEWS": 10, "SAD": 12, "SURPRISE": 14, "UMANG": 15, "WIKI": 16,
}
# Per-target-language voice availability, keyed by this pipeline's ISO codes.
# None = that gender doesn't exist in the model (ml/ta/te are female-only).
VITS_RASA_VOICES_BY_LANG: dict[str, dict] = {
    "as": {"F": 0, "M": 1},
    "bn": {"F": 2, "M": 3},
    "kn": {"F": 8, "M": 9},
    "ml": {"F": 11, "M": None},
    "mr": {"F": 12, "M": 13},
    "pa": {"F": 15, "M": 16},
    "ta": {"F": 18, "M": None},
    "te": {"F": 19, "M": None},
}
VITS_RASA_UNSUPPORTED: frozenset = frozenset({"hi", "gu", "or"})

# pyannote speaker diarization (multi-speaker dubbing's segmentation stage). BOTH repos
# are gated on HF - accept terms on each model page with the same HF_TOKEN account:
#   huggingface.co/pyannote/speaker-diarization-3.1
#   huggingface.co/pyannote/segmentation-3.0  (pulled in by the pipeline above)
PYANNOTE_DIARIZATION_HF_PATH = "pyannote/speaker-diarization-3.1"


def get_language(code: str) -> LanguageInfo:
    code = code.lower().strip()
    if code not in LANGUAGES:
        raise ValueError(
            f"Unknown language code '{code}'. Supported: {sorted(LANGUAGES.keys())}"
        )
    return LANGUAGES[code]


def all_codes() -> list[str]:
    return list(LANGUAGES.keys())


def display_name(code: str) -> str:
    return get_language(code).name


In [ ]:
%%writefile /kaggle/working/pipeline_v3/common/phonemes.py
"""
common/phonemes.py
==================
THE canonical grapheme-to-phoneme counter for pipeline_v3. Every module that writes a
phoneme budget into a label, and every module that scores a generation against one, must
import `count_phonemes` from here and from nowhere else.

WHY THIS MODULE EXISTS (read this before changing anything in it)
-----------------------------------------------------------------
The previous implementation lived in `translation/duration_predictor.phonemize_text` and
ended like this::

    except Exception as e:
        logger.debug("Phonemization failed ...; falling back to characters.")
    return [c for c in text if not c.isspace()]

That fallback fired for the **entire** corpus generation run, because espeak-ng (a system
binary, installed separately from the `phonemizer` pip package) was not present in that
Kaggle session. It logged at DEBUG, so nothing surfaced. The result: every `n_phonemes`
label in the base training corpus is a **non-space character count**, verified at 100.0%
over 1,289 locally-held rows with a chars/n_phonemes ratio of exactly 1.000 (min = max).

It got worse. A later session (length augmentation, 2026-07-26) *did* have espeak-ng, so
those rows are labelled in **real phonemes** — 8.4% coincidental agreement with character
counts, ratio spread 0.198-1.500. The corpus therefore carries two mutually incompatible
rulers under one prompt token, `[Target Phonemes: N]`, teaching the model two
contradictory tasks. A model asked to fit N of one unit and N of another cannot reach
slope 1.0 on either; it can only split the difference.

So this module enforces three things the old one did not:

1. **No silent fallback, ever.** A phonemization failure raises `G2PUnavailable` or
   `PhonemizationError`. Label-writing code must never degrade quietly, because a
   degraded label is indistinguishable from a good one downstream. Inference code that
   legitimately needs to survive a bad string catches the exception *explicitly* and
   records the degradation (see "degrade, don't crash" — but degrade *visibly*).

2. **A preflight that proves the output is phonemes, not passthrough.** Checking that
   espeak-ng is importable is not sufficient — the failure mode we actually hit produces
   plausible-looking output. `assert_g2p_available()` phonemizes a canary string in each
   language and asserts the returned symbols are not simply the input's own characters.
   That is the check that would have caught this on day one.

3. **A ruler identifier stamped into every artifact.** `ruler_id()` returns a string like
   ``phonemes:espeak-ng-1.51``. Dataset rows, eval reports, and run manifests all carry
   it, so "which ruler produced this number" is a grep, not a forensic exercise.

WHAT COUNTS AS ONE PHONEME
---------------------------
espeak-ng's IPA output carries symbols that are not sounds. We normalise before counting:

- **Stress marks** (``ˈ`` primary, ``ˌ`` secondary) are suprasegmental — they mark which
  syllable is emphasised, not an additional sound. Stripped.
- **Length marks** (``ː``) modify the preceding vowel's duration and stay attached to it,
  so ``aː`` is one (long) phoneme, not two. Kept attached — which is correct for our
  purpose, since duration is exactly what we are proxying.
- **Language-switch tags** (``(en)``, ``(hi)``) are emitted when espeak detects a foreign
  word — typically English brand names inside Indic text. They are markup. Stripped.
- **Tie bars** (``͡``) join affricates into one segment. Kept attached.

These choices are asymmetric-safe: they can only ever be wrong by a constant per language,
and non-negotiable #3 (same function for labels and scores) means a constant offset
cancels. What must never happen is two *different* normalisations in the same project.
"""

from __future__ import annotations

import functools
import logging
import re
from typing import Iterable, Optional, Sequence

from common.languages import LANGUAGES, get_language

logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------------------
# Ruler identifiers. These are written into datasets and reports; treat them as a stable
# public vocabulary, not as free text.
# ---------------------------------------------------------------------------------------

RULER_PHONEMES = "phonemes:espeak-ng"
RULER_CHARS = "chars:non-space"
RULER_UNKNOWN = "unknown"


class G2PUnavailable(RuntimeError):
    """espeak-ng and/or the `phonemizer` package is missing or non-functional.

    Raised by the preflight and by `phonemize()`. This is deliberately fatal: every
    caller in the label-writing path would otherwise produce a corpus that looks correct
    and is measured in the wrong unit.
    """


class PhonemizationError(RuntimeError):
    """espeak-ng is present and working, but this specific string could not be converted."""


_INSTALL_HINT = (
    "espeak-ng is a SYSTEM package and is NOT installed by `pip install phonemizer`.\n"
    "  Kaggle / Debian / Ubuntu:  apt-get -qq update && apt-get -qq install -y espeak-ng\n"
    "  macOS:                     brew install espeak-ng\n"
    "  Windows:                   winget install --id eSpeak-NG.eSpeak-NG   (then set\n"
    "                             PHONEMIZER_ESPEAK_LIBRARY to the installed libespeak-ng.dll)\n"
    "  Then:                      pip install phonemizer\n"
    "Verify with: python -c \"from common.phonemes import assert_g2p_available;"
    " assert_g2p_available()\""
)

# Suprasegmental / markup symbols that are not themselves sounds.
_STRESS_MARKS = "ˈˌ"          # ˈ ˌ
_LANG_SWITCH_RE = re.compile(r"\([a-z]{2,3}\)")   # (en), (hi), ...
_UNDERTIE = "‿"

# Unicode names several scripts by an older label than the one the language table uses.
_UNICODE_SCRIPT_NAME = {"odia": "ORIYA"}


def _script_token(language_iso_code: str) -> str:
    """The token that appears in unicodedata.name() for this language's script."""
    script = get_language(language_iso_code).script.lower()
    return _UNICODE_SCRIPT_NAME.get(script, script.split()[0].upper())


# ---------------------------------------------------------------------------------------
# Orthographic normalisation (AI4Bharat IndicNLP)
# ---------------------------------------------------------------------------------------

@functools.lru_cache(maxsize=16)
def _normalizer(language_iso_code: str):
    """Per-language IndicNLP normaliser, or None if the library is absent."""
    try:
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    except ImportError:
        return None
    return IndicNormalizerFactory().get_normalizer(language_iso_code)


# Set once, the first time normalisation is skipped, so the absence is stated rather than
# inferred from a slightly-off number three stages downstream.
_NORMALIZER_WARNED: set = set()


@functools.lru_cache(maxsize=1)
def _recomposition_map() -> dict:
    """decomposed-sequence -> precomposed-character, for Indic nukta letters.

    Built from `unicodedata` rather than hand-listed, so it cannot drift from the standard
    and needs no maintenance when a script is added.

    WHY IT IS NEEDED. IndicNLP canonicalises *toward the decomposed form*: `য়` U+09DF
    becomes U+09AF + U+09BC (YA + NUKTA). espeak-ng's rule files are evidently written
    against the *precomposed* letters. Measured on this corpus: normalisation rewrites
    50.8% of Assamese rows, almost all of them this substitution, and Assamese
    phonemes-per-character then jumps 1.147 -> 1.604 while Bengali — same script, same
    substitution on 36.3% of its rows — barely moves (1.032 -> 1.020). Assamese and Bengali
    are phonologically close and orthographically shared; a 57% divergence between them is
    not a better measurement, it is espeak's `as` voice failing to parse a sequence its
    `bn` voice handles.

    These characters cannot be recomposed by `unicodedata.normalize("NFC", ...)`, because
    Indic nukta letters are Unicode *composition exclusions* — NFC deliberately leaves them
    decomposed. Hence an explicit map.

    So: take IndicNLP's genuine cleanups (ZWJ/ZWNJ removal, punctuation canonicalisation,
    Malayalam chillu handling) and then put the nukta letters back into the encoding the
    G2P actually recognises.
    """
    import unicodedata
    mapping = {}
    # Devanagari, Bengali, Gurmukhi, Gujarati, Oriya, Tamil, Telugu, Kannada, Malayalam
    for cp in range(0x0900, 0x0E00):
        ch = chr(cp)
        decomp = unicodedata.decomposition(ch)
        if not decomp or decomp.startswith("<"):   # skip compatibility decompositions
            continue
        try:
            seq = "".join(chr(int(p, 16)) for p in decomp.split())
        except ValueError:
            continue
        mapping[seq] = ch
    return mapping


def recompose_indic(text: str) -> str:
    """Restores precomposed Indic nukta letters after IndicNLP normalisation."""
    for seq, ch in _recomposition_map().items():
        if seq in text:
            text = text.replace(seq, ch)
    return text


def normalize_indic(text: str, language_iso_code: str) -> str:
    """Canonicalises Indic text before G2P.

    Indic scripts encode the same grapheme several ways, and espeak-ng only has rules for
    one of them. Verified on this machine:

        क़  U+0958 (precomposed)      -> U+0915 U+093C  (KA + NUKTA)
        ড়  U+09DC (precomposed)      -> U+09A1 U+09BC  (DDA + NUKTA)
        क्‍ष  with U+200D ZWJ            -> ZWJ removed

    Fed the precomposed or ZWJ-bearing form, espeak either skips the codepoint or emits
    something arbitrary — silently, and only for the subset of rows that happen to use that
    encoding. That is a per-row error concentrated in exactly the words most likely to be
    loanwords and proper nouns, which is worse than a uniform bias because it cannot be
    calibrated away.

    Normalising is a strict improvement and costs nothing: on already-canonical text it is
    a no-op (verified across all 11 languages). It runs inside `phonemize`, so labels and
    scores are normalised identically — the same-function rule (non-negotiable #3) extends
    to preprocessing, not just to the G2P call.
    """
    n = _normalizer(language_iso_code)
    if n is None:
        if "warned" not in _NORMALIZER_WARNED:
            _NORMALIZER_WARNED.add("warned")
            logger.warning(
                "indic-nlp-library is not installed — phoneme counts will be taken on "
                "UN-normalised text. Precomposed nukta forms and ZWJ sequences will be "
                "mis-phonemized for a minority of rows. Install with: "
                "pip install indic-nlp-library"
            )
        return text
    return recompose_indic(n.normalize(text))


# ---------------------------------------------------------------------------------------
# Backend access
# ---------------------------------------------------------------------------------------

@functools.lru_cache(maxsize=1)
def _backend_version() -> str:
    """Returns the espeak-ng version string, or raises G2PUnavailable.

    Cached because it shells into the espeak library, and callers ask for it once per
    artifact written.
    """
    try:
        from phonemizer.backend.espeak.wrapper import EspeakWrapper
    except ImportError as e:
        raise G2PUnavailable(f"`phonemizer` is not importable ({e}).\n{_INSTALL_HINT}") from e

    try:
        version = EspeakWrapper().version
    except Exception as e:  # noqa: BLE001 - any failure here means the shared library is unusable
        raise G2PUnavailable(
            f"`phonemizer` imported but the espeak-ng shared library is unusable ({e}).\n"
            f"{_INSTALL_HINT}"
        ) from e

    # Normalise: some phonemizer builds return a tuple like (1, 50) rather than "1.50".
    # `str()` on that yields "(1, 50)" — which embeds a COMMA and a space into a string
    # that gets stamped into every dataset row and every report header, and would corrupt
    # any CSV column it ever lands in. A provenance field that can break its own container
    # is not provenance.
    if isinstance(version, (tuple, list)):
        version = ".".join(str(p) for p in version)
    return re.sub(r"[\s,]+", "", str(version))


def ruler_id() -> str:
    """The identifier stamped into every dataset row and eval report this module touches.

    Raises G2PUnavailable rather than returning a placeholder — a report that cannot name
    its ruler must not be written at all.
    """
    return f"{RULER_PHONEMES}-{_backend_version()}"


# ---------------------------------------------------------------------------------------
# Core conversion
# ---------------------------------------------------------------------------------------

def _normalise_tokens(raw: str) -> list[str]:
    """Turns espeak's separated output into a list of countable phoneme symbols."""
    raw = _LANG_SWITCH_RE.sub(" ", raw)
    raw = raw.replace("|", " ").replace(_UNDERTIE, " ")
    tokens = []
    for tok in raw.split():
        tok = tok.strip(_STRESS_MARKS)
        # A token that was *only* stress marks collapses to empty and is not a sound.
        if tok:
            tokens.append(tok)
    return tokens


def phonemize(text: str, language_iso_code: str) -> list[str]:
    """Converts `text` to a list of IPA phoneme symbols. Never falls back.

    Raises:
        G2PUnavailable: espeak-ng / phonemizer is missing or broken.
        PhonemizationError: the backend works but produced nothing for this input.
    """
    text = (text or "").strip()
    if not text:
        return []

    _backend_version()  # raises G2PUnavailable with the install hint if the backend is dead

    from phonemizer import phonemize as _ph
    from phonemizer.separator import Separator

    lang = get_language(language_iso_code)
    text = normalize_indic(text, language_iso_code)
    try:
        out = _ph(
            text,
            language=lang.espeak_code,
            backend="espeak",
            separator=Separator(phone=" ", word=" | "),
            strip=True,
            njobs=1,
        )
    except Exception as e:  # noqa: BLE001
        raise PhonemizationError(
            f"espeak-ng failed on {language_iso_code} input {text[:60]!r}: {e}"
        ) from e

    tokens = _normalise_tokens(out)
    if not tokens:
        raise PhonemizationError(
            f"espeak-ng returned no phonemes for {language_iso_code} input {text[:60]!r}. "
            f"Raw backend output was {out[:80]!r}."
        )
    return tokens


def phonemize_many(texts: Sequence[str], language_iso_code: str) -> list[list[str]]:
    """Batched `phonemize`, one espeak call for the whole list.

    Roughly an order of magnitude faster than looping — which matters, because relabelling
    the 53,350-row corpus one string at a time is a multi-hour job and a batched pass is
    minutes. Empty inputs map to empty lists; a backend failure on the batch raises, so a
    partially-phonemized corpus is never written.
    """
    if not texts:
        return []

    _backend_version()

    from phonemizer import phonemize as _ph
    from phonemizer.separator import Separator

    lang = get_language(language_iso_code)
    cleaned = [normalize_indic((t or "").strip(), language_iso_code) for t in texts]
    try:
        out = _ph(
            cleaned,
            language=lang.espeak_code,
            backend="espeak",
            separator=Separator(phone=" ", word=" | "),
            strip=True,
            njobs=1,
        )
    except Exception as e:  # noqa: BLE001
        raise PhonemizationError(
            f"espeak-ng failed on a batch of {len(texts)} {language_iso_code} strings: {e}"
        ) from e

    if isinstance(out, str):  # phonemizer collapses a 1-element list to a bare string
        out = [out]
    return [_normalise_tokens(o) for o in out]


@functools.lru_cache(maxsize=200_000)
def count_phonemes(text: str, language_iso_code: str) -> int:
    """The one function that defines "how many phonemes is this". Labels and scores both
    call it, which is what makes their numbers comparable (non-negotiable #3)."""
    return len(phonemize(text, language_iso_code))


def phoneme_inventory(texts: Iterable[str], language_iso_code: str) -> "Counter":
    """The distribution of phoneme symbols a language's G2P actually produces.

    This is a *validation* instrument, not a pipeline metric — the pipeline only ever needs
    the per-sentence count. But a count cannot tell you whether the symbols being counted
    are phonemes at all, and an inventory can, in one glance.

    Worked example of what it catches: epitran's Marathi renders `कॅल्शियम` as `kəॅlɕijmə`,
    leaking U+0945 DEVANAGARI VOWEL SIGN CANDRA E — a *source script* character — straight
    into its own IPA output, because that codepoint has no entry in its map. The count still
    comes out looking reasonable. The inventory makes it obvious.
    """
    from collections import Counter
    inv = Counter()
    for toks in phonemize_many(list(texts), language_iso_code):
        inv.update(toks)
    return inv


def validate_inventory(inventory: "Counter", language_iso_code: str) -> list[str]:
    """Returns a list of problems found in a phoneme inventory; empty means clean.

    The check that matters: a symbol containing a character from the language's OWN script
    is not a phoneme — it is an unmapped source character that the G2P passed through
    untranslated. Its presence proves the converter has a hole, and tells you exactly which
    grapheme fell in.
    """
    import unicodedata

    token = _script_token(language_iso_code)
    problems = []
    total = sum(inventory.values()) or 1
    for sym, n in inventory.most_common():
        leaked = [c for c in sym if token in unicodedata.name(c, "")]
        if leaked:
            names = ", ".join(f"U+{ord(c):04X} {unicodedata.name(c, '?')}" for c in leaked)
            problems.append(
                f"{sym!r} ({n} occurrences, {100 * n / total:.2f}%) contains untranslated "
                f"source-script characters: {names}"
            )
    return problems


def count_chars(text: str) -> int:
    """Non-space character count — the *wrong* ruler, defined here explicitly so audit
    code can name it and detect it rather than reimplementing it three times."""
    return len([c for c in (text or "") if not c.isspace()])


# ---------------------------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------------------------

# Short canaries in each language's own script. Any real sentence works; these are kept
# tiny so the preflight costs milliseconds.
_CANARIES: dict[str, str] = {
    "hi": "दूध में कैल्शियम है",
    "bn": "দুধে ক্যালসিয়াম আছে",
    "mr": "दुधात कॅल्शियम आहे",
    "gu": "દૂધમાં કેલ્શિયમ છે",
    "pa": "ਦੁੱਧ ਵਿੱਚ ਕੈਲਸ਼ੀਅਮ ਹੈ",
    "ta": "பாலில் கால்சியம் உள்ளது",
    "te": "పాలలో కాల్షియం ఉంది",
    "kn": "ಹಾಲಿನಲ್ಲಿ ಕ್ಯಾಲ್ಸಿಯಂ ಇದೆ",
    "ml": "പാലിൽ കാൽസ്യം ഉണ്ട്",
    "or": "ଦୁଧରେ କ୍ୟାଲସିୟମ ଅଛି",
    "as": "গাখীৰত কেলচিয়াম আছে",
}


def assert_g2p_available(languages: Optional[Iterable[str]] = None, verbose: bool = True) -> dict:
    """Preflight. Call this at the top of every notebook that writes labels or scores them.

    Checks three things, in increasing order of strictness:

    1. `phonemizer` imports and the espeak-ng shared library loads.
    2. Every requested language produces non-empty output.
    3. **The output is actually phonemes, not the input's own characters.** This is the
       check that matters. A backend that silently passes text through, or a caller that
       silently substitutes a character split, both produce plausible non-empty output —
       and that is precisely the failure that mislabelled this project's corpus. We assert
       that at least one returned symbol does not appear in the source string, which is
       guaranteed true for any Indic script rendered to IPA and false for passthrough.

    Returns a manifest dict suitable for writing next to any artifact produced afterwards.

    Raises:
        G2PUnavailable: with the install hint, if any check fails.
    """
    codes = list(languages) if languages is not None else list(LANGUAGES.keys())
    version = _backend_version()   # raises with install hint

    report: dict[str, dict] = {}
    failures: list[str] = []

    for code in codes:
        canary = _CANARIES.get(code)
        if canary is None:
            failures.append(f"{code}: no canary string defined in common/phonemes.py")
            continue
        try:
            tokens = phonemize(canary, code)
        except (G2PUnavailable, PhonemizationError) as e:
            failures.append(f"{code}: {e}")
            continue

        source_chars = set(canary)
        novel = [t for t in tokens if not set(t) <= source_chars]
        looks_like_passthrough = not novel

        # Partial leaks: individual unmapped graphemes riding through into the IPA. The
        # passthrough test above only catches total failure; this catches the holes.
        from collections import Counter
        leaks = validate_inventory(Counter(tokens), code)

        report[code] = {
            "voice": get_language(code).espeak_code,
            "n_phonemes": len(tokens),
            "n_chars": count_chars(canary),
            "sample": " ".join(tokens[:12]),
            "passthrough": looks_like_passthrough,
            "normalized": _normalizer(code) is not None,
            "leaks": leaks,
        }
        if looks_like_passthrough:
            failures.append(
                f"{code}: espeak returned symbols drawn entirely from the input's own "
                f"characters ({' '.join(tokens[:10])}) — this is a passthrough or a "
                f"character-split fallback, NOT phonemization."
            )
        for lk in leaks:
            failures.append(f"{code}: untranslated grapheme in G2P output — {lk}")

    if failures:
        raise G2PUnavailable(
            "G2P preflight FAILED — do not generate labels or scores in this session.\n"
            + "\n".join(f"  - {f}" for f in failures)
            + "\n\n" + _INSTALL_HINT
        )

    manifest = {
        "ruler": ruler_id(),
        "espeak_version": version,
        "languages": report,
    }
    if verbose:
        logger.info("G2P preflight PASSED — ruler=%s", manifest["ruler"])
        for code, r in report.items():
            logger.info(
                "  %-3s voice=%-3s %3d phonemes / %3d chars (ratio %.3f)  %s",
                code, r["voice"], r["n_phonemes"], r["n_chars"],
                r["n_phonemes"] / max(r["n_chars"], 1), r["sample"],
            )
    return manifest


if __name__ == "__main__":  # `python -m common.phonemes` as a standalone preflight
    import json
    import sys

    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    try:
        print(json.dumps(assert_g2p_available(), ensure_ascii=False, indent=2))
    except G2PUnavailable as e:
        print(str(e), file=sys.stderr)
        raise SystemExit(1)


In [ ]:
%%writefile /kaggle/working/pipeline_v3/tools/__init__.py
"""Standalone diagnostic and data-repair utilities for pipeline_v3.

These are not part of the runtime pipeline. They exist to answer questions about
artifacts the pipeline has already produced — which ruler labelled a corpus, whether a
checkpoint can be salvaged, whether a relabel actually changed what it claimed to.
"""


In [ ]:
%%writefile /kaggle/working/pipeline_v3/tools/ruler_audit.py
"""
tools/ruler_audit.py
====================
Answers two questions about a training/validation corpus, and one decision that follows
from them.

QUESTION 1 — "Which ruler wrote these labels?"
----------------------------------------------
For every row, `n_phonemes` is compared against both candidate rulers computed from the
row's own target text: the non-space character count, and the true espeak-ng phoneme
count. Whichever matches is the ruler that wrote the label. A corpus where different rows
answer differently is **mixed**, which is worse than a corpus that is uniformly wrong: a
uniformly-wrong corpus teaches one consistent (if mislabelled) task, while a mixed corpus
teaches two contradictory tasks under the same prompt token and the model can only split
the difference between them.

This project's corpus is mixed. `training/dataset_generator.py` wrote character counts
(its espeak fallback fired for the whole run); `training/length_augmentation.py`, run in a
later session that did have espeak-ng, wrote real phoneme counts. Both used the prompt
token `[Target Phonemes: N]`.

QUESTION 2 — "Is chars -> phonemes a tight enough map to rescale instead of retrain?"
-------------------------------------------------------------------------------------
This is the load-bearing question and the reason this tool exists before any GPU is
booked. A model trained on character budgets learned a real, usable capability — it just
learned it in the wrong unit. If, *within a language*, phonemes are a near-constant
multiple of characters, then the existing checkpoint is salvageable with no retraining at
all: at dub time, convert the phoneme budget you want into the character budget the model
was actually taught, prompt with that, and the model lands where you meant.

If instead the ratio is noisy within a language, then the character label was only weakly
informative about duration, the conditioning signal the model received was correspondingly
noisy, and no amount of inference-time arithmetic recovers it — the corpus must be
relabelled and the model retrained.

THE DECISION RULE, PRE-REGISTERED
----------------------------------
Written down here, before the numbers are seen, so the answer cannot be rationalised after
the fact (Gate 0 of the training doctrine).

Per language, over the phonemes/chars ratio:

    CV = std(ratio) / mean(ratio)          # coefficient of variation, unitless

  - **CV <= 0.08 for all 11 languages** -> SALVAGE. A per-language constant rescale
    recovers the capability. Retraining buys accuracy, not correctness. Ship the rescale,
    relabel the corpus for the *next* run, do not spend a session on it now.

  - **CV > 0.15 for any language** -> RETRAIN that language's supervision. The label was
    too weakly coupled to duration to have taught budget obedience.

  - **In between** -> rescale, and measure the residual length error on the probe. The
    rescale is free; whether it is sufficient is an empirical question the length-response
    probe answers directly.

The 0.08 threshold is not arbitrary: the fine-tuned model's own reported length error is
~10%, so a rescale whose own noise floor is under ~8% is not the binding constraint. A
rescale noisier than the model's existing error would be adding more error than it removes.

USAGE
-----
    python -m tools.ruler_audit --jsonl data/translation_dataset/train.jsonl \\
                                --out research/ruler_audit_train.json

    # audit several at once, e.g. train + val + the augmented file
    python -m tools.ruler_audit --jsonl data/.../train.jsonl data/.../val.jsonl \\
                                --out research/ruler_audit.json

Exit code is 1 if any audited file is mixed-ruler or character-ruled, so this can gate a
notebook cell rather than merely inform one.
"""

from __future__ import annotations

import argparse
import json
import logging
import math
import statistics
import sys
from collections import Counter, defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.languages import LANGUAGES  # noqa: E402
from common.phonemes import (  # noqa: E402
    RULER_CHARS, RULER_PHONEMES, RULER_UNKNOWN, assert_g2p_available, count_chars,
    phonemize_many, ruler_id, validate_inventory,
)

logger = logging.getLogger("ruler_audit")

# Pre-registered thresholds — see module docstring. Do not tune these to make a result
# come out the way you want; change them only with a written justification.
CV_SALVAGE_MAX = 0.08
CV_RETRAIN_MIN = 0.15


def read_jsonl(path: str) -> list[dict]:
    rows, bad = [], 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
    if bad:
        logger.warning("%s: %d unparseable lines skipped", path, bad)
    return rows


def _fit_through_origin(xs: list[float], ys: list[float]) -> tuple[float, float]:
    """OLS slope for y = k*x with no intercept, plus R^2 about that model.

    Through the origin rather than with a free intercept because the relationship is
    physically proportional — a sentence with no characters has no phonemes — and a free
    intercept would let a bad fit hide behind an offset.
    """
    sxx = sum(x * x for x in xs)
    if sxx == 0:
        return float("nan"), float("nan")
    k = sum(x * y for x, y in zip(xs, ys)) / sxx
    ss_res = sum((y - k * x) ** 2 for x, y in zip(xs, ys))
    my = sum(ys) / len(ys)
    ss_tot = sum((y - my) ** 2 for y in ys)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return k, r2


def audit_file(path: str, max_rows_per_lang: int | None = None) -> dict:
    rows = read_jsonl(path)
    by_lang: dict[str, list[dict]] = defaultdict(list)
    for r in rows:
        by_lang[r.get("language", "unknown")].append(r)

    per_lang: dict[str, dict] = {}
    for lang in sorted(by_lang):
        lang_rows = by_lang[lang]
        if max_rows_per_lang:
            lang_rows = lang_rows[:max_rows_per_lang]
        if lang not in LANGUAGES:
            logger.warning("skipping unknown language '%s' (%d rows)", lang, len(lang_rows))
            continue

        targets = [(r.get("target") or r.get("completion") or "") for r in lang_rows]
        labels = [int(r.get("n_phonemes") or 0) for r in lang_rows]

        # One batched espeak call per language rather than one per row.
        tokenized = phonemize_many(targets, lang)
        true_ph = [len(toks) for toks in tokenized]
        chars = [count_chars(t) for t in targets]

        # The phoneme inventory this language's G2P actually produced, over the whole
        # corpus. A count cannot tell you whether the things being counted are phonemes;
        # the inventory can. `leaks` names any source-script grapheme that rode through
        # untranslated — a hole in the converter, with the exact codepoint that fell in.
        inventory = Counter()
        for toks in tokenized:
            inventory.update(toks)
        leaks = validate_inventory(inventory, lang)

        keep = [i for i in range(len(labels)) if labels[i] > 0 and chars[i] > 0 and true_ph[i] > 0]
        if not keep:
            continue
        labels = [labels[i] for i in keep]
        true_ph = [true_ph[i] for i in keep]
        chars = [chars[i] for i in keep]
        aug_flags = [("augmentation" in lang_rows[i]) for i in keep]

        n = len(labels)
        match_chars = sum(1 for i in range(n) if labels[i] == chars[i])
        match_phon = sum(1 for i in range(n) if labels[i] == true_ph[i])

        ratios = [true_ph[i] / chars[i] for i in range(n)]
        mean_ratio = statistics.fmean(ratios)
        # Sample stdev (n-1). The old eval harness used the population divisor, which
        # understates spread on small samples — exactly where an audit must not be
        # optimistic.
        sd_ratio = statistics.stdev(ratios) if n > 1 else 0.0
        cv = sd_ratio / mean_ratio if mean_ratio else float("nan")
        k, r2 = _fit_through_origin([float(c) for c in chars], [float(p) for p in true_ph])

        # How wrong is the label, in the unit that matters?
        label_err = [abs(labels[i] - true_ph[i]) / true_ph[i] for i in range(n)]

        frac_chars = match_chars / n
        frac_phon = match_phon / n
        if frac_chars >= 0.95:
            ruler = RULER_CHARS
        elif frac_phon >= 0.95:
            ruler = RULER_PHONEMES
        elif frac_chars + frac_phon >= 0.95:
            ruler = "MIXED"
        else:
            ruler = RULER_UNKNOWN

        per_lang[lang] = {
            "n": n,
            "n_augmented": sum(aug_flags),
            "ruler": ruler,
            "inventory_size": len(inventory),
            "inventory_top20": inventory.most_common(20),
            "inventory_leaks": leaks,
            "frac_label_eq_chars": round(frac_chars, 4),
            "frac_label_eq_phonemes": round(frac_phon, 4),
            "phonemes_per_char_mean": round(mean_ratio, 4),
            "phonemes_per_char_sd": round(sd_ratio, 4),
            "phonemes_per_char_cv": round(cv, 4),
            "ols_k_through_origin": round(k, 4),
            "ols_r2": round(r2, 4),
            "label_rel_error_vs_true_phonemes_mean": round(statistics.fmean(label_err), 4),
            "verdict": (
                "SALVAGE" if cv <= CV_SALVAGE_MAX
                else "RETRAIN" if cv > CV_RETRAIN_MIN
                else "RESCALE_THEN_MEASURE"
            ),
        }

    rulers = {v["ruler"] for v in per_lang.values()}
    if rulers == {RULER_PHONEMES}:
        corpus_ruler = RULER_PHONEMES
    elif rulers == {RULER_CHARS}:
        corpus_ruler = RULER_CHARS
    else:
        corpus_ruler = "MIXED"

    verdicts = {v["verdict"] for v in per_lang.values()}
    if verdicts == {"SALVAGE"}:
        corpus_verdict = "SALVAGE"
    elif "RETRAIN" in verdicts:
        corpus_verdict = "RETRAIN"
    else:
        corpus_verdict = "RESCALE_THEN_MEASURE"

    return {
        "file": path,
        "n_rows": len(rows),
        "scoring_ruler": ruler_id(),
        "corpus_ruler": corpus_ruler,
        "corpus_verdict": corpus_verdict,
        "per_language": per_lang,
    }


def _print_table(result: dict) -> None:
    print(f"\n=== {result['file']} ===")
    print(f"rows={result['n_rows']}  corpus_ruler={result['corpus_ruler']}  "
          f"verdict={result['corpus_verdict']}")
    print(f"scored with: {result['scoring_ruler']}\n")
    hdr = (f"{'lang':<5}{'n':>6}{'aug':>6}  {'ruler':<16}{'=chars':>8}{'=phon':>8}"
           f"{'ph/char':>9}{'CV':>8}{'R2':>7}{'lblErr':>8}  verdict")
    print(hdr)
    print("-" * len(hdr))
    for lang, v in result["per_language"].items():
        print(f"{lang:<5}{v['n']:>6}{v['n_augmented']:>6}  {v['ruler']:<16}"
              f"{v['frac_label_eq_chars']:>8.2f}{v['frac_label_eq_phonemes']:>8.2f}"
              f"{v['phonemes_per_char_mean']:>9.3f}{v['phonemes_per_char_cv']:>8.3f}"
              f"{v['ols_r2']:>7.3f}{v['label_rel_error_vs_true_phonemes_mean']:>8.3f}"
              f"  {v['verdict']}")

    print("\nPhoneme inventories (validation, not a pipeline metric):")
    for lang, v in result["per_language"].items():
        top = " ".join(s for s, _ in v["inventory_top20"][:14])
        print(f"  {lang}: {v['inventory_size']:>3} distinct | {top}")
        for lk in v["inventory_leaks"]:
            print(f"      !! LEAK {lk}")
    if not any(v["inventory_leaks"] for v in result["per_language"].values()):
        print("  no untranslated source-script graphemes in any language's output.")


def _print_decision(results: list[dict]) -> None:
    print("\n" + "=" * 78)
    print("DECISION")
    print("=" * 78)
    verdicts = {r["corpus_verdict"] for r in results}
    if "RETRAIN" in verdicts:
        print("RETRAIN. At least one language's phoneme/char ratio varies more than 15%")
        print("within the language, so the character label was too weakly coupled to")
        print("duration to have taught budget obedience. Relabel the corpus with")
        print("tools/relabel_dataset.py and retrain. A rescale cannot recover this.")
    elif verdicts == {"SALVAGE"}:
        print("SALVAGE. Every language's phoneme/char ratio is stable to within 8%.")
        print("The existing checkpoint learned the right capability in the wrong unit.")
        print("Apply the per-language `ols_k_through_origin` as an inference-time budget")
        print("conversion (phoneme budget / k = the character budget the model was")
        print("taught) and re-run the length-response probe to confirm. Relabel the")
        print("corpus for the NEXT training run, not this one.")
    else:
        print("RESCALE, THEN MEASURE. The ratio is stable enough that the conversion is")
        print("worth applying and free, but not so stable that it is guaranteed")
        print("sufficient. Apply the rescale, run the length-response probe, and let the")
        print("residual slope decide whether a relabelled retrain is still needed.")
    print("\nPer-language conversion constants (phoneme_budget / k = char_budget to prompt):")
    for r in results:
        for lang, v in r["per_language"].items():
            print(f"  {lang}: k = {v['ols_k_through_origin']:.4f}  (R2 {v['ols_r2']:.3f}, "
                  f"CV {v['phonemes_per_char_cv']:.3f})")
        break  # constants are a property of the language, not the file


def main() -> int:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--jsonl", nargs="+", required=True, help="One or more corpus files to audit.")
    p.add_argument("--out", default=None, help="Write the full JSON report here.")
    p.add_argument("--max_rows_per_lang", type=int, default=None,
                   help="Subsample for a fast pass; omit to audit every row.")
    args = p.parse_args()

    # Refuse to audit at all without a verified phonemizer — an audit that silently used
    # the character fallback would 'prove' the corpus was correctly labelled.
    assert_g2p_available()

    results = [audit_file(path, args.max_rows_per_lang) for path in args.jsonl]
    for r in results:
        _print_table(r)
    _print_decision(results)

    if args.out:
        Path(args.out).parent.mkdir(parents=True, exist_ok=True)
        Path(args.out).write_text(json.dumps(results, ensure_ascii=False, indent=2),
                                  encoding="utf-8")
        print(f"\nWrote {args.out}")

    bad = [r for r in results if r["corpus_ruler"] != RULER_PHONEMES]
    if bad:
        print(f"\nFAIL: {len(bad)} file(s) are not phoneme-ruled: "
              f"{', '.join(r['file'] for r in bad)}")
        return 1
    print("\nPASS: every audited file is phoneme-ruled.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/pipeline_v3/tools/relabel_dataset.py
"""
tools/relabel_dataset.py
========================
Rewrites a corpus's `n_phonemes` labels and `[Target Phonemes: N]` prompts using the
canonical counter in `common/phonemes.py`, and stamps enough provenance into the output
that the mislabelling this repairs can never recur silently.

WHAT IT REPAIRS
---------------
1. **The label itself.** `n_phonemes` is recomputed from the row's own target text with
   espeak-ng. Every row gets `ruler` and `n_phonemes_legacy` fields, so the change is
   auditable and reversible rather than destructive.

2. **The prompt.** `[Target Phonemes: N]` is regenerated from the new N. Leaving the
   prompt stale while fixing the label would be strictly worse than doing nothing — the
   model reads the prompt, and the loss is computed against a completion whose length now
   disagrees with it.

3. **The augmentation direction gate, which was evaluated in mixed units.**
   `length_augmentation.py` admitted a paraphrase if
   `(variant_phonemes - source_phonemes) / source_phonemes` moved >= 10% in the intended
   direction. But `variant_phonemes` was a real phoneme count while `source_phonemes` was
   read from the base row's `n_phonemes`, which was a character count. That expression is
   not a length-change measurement; it is a comparison between two different units, and
   its sign and magnitude are both untrustworthy.

   So every augmented row is re-tested with both sides measured in real phonemes. Variants
   that no longer clear the threshold were admitted on a broken comparison and are dropped
   by default — a "compressed" example that did not actually compress teaches the model
   that the budget token means nothing, which is the precise opposite of the objective.
   Use `--keep_failed_augmentation` to retain and flag them instead of dropping.

WHAT IT DELIBERATELY DOES NOT DO
---------------------------------
It does not re-run the *semantic* gate. That gate compared the paraphrase against the
human reference with a multilingual embedder and its 0.80 threshold is unit-free, so it
was unaffected by the ruler bug and its verdicts still stand. Re-running it would need a
GPU and would change nothing.

USAGE
-----
    python -m tools.relabel_dataset \\
        --in  data/translation_dataset/train.jsonl \\
        --out data/translation_dataset/train.phonemes.jsonl

    # then confirm:
    python -m tools.ruler_audit --jsonl data/translation_dataset/train.phonemes.jsonl

A sidecar `<out>.manifest.json` records the espeak version, the ruler id, per-language
before/after label statistics, and the augmentation re-validation tally.
"""

from __future__ import annotations

import argparse
import json
import logging
import statistics
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.languages import LANGUAGES  # noqa: E402
from common.phonemes import assert_g2p_available, count_chars, phonemize_many, ruler_id  # noqa: E402

logger = logging.getLogger("relabel_dataset")

PROMPT_TEMPLATE = '[Translate to {language}] [Target Phonemes: {n_phonemes}] "{english}"'

# Must match training/length_augmentation.py's default. Imported as a literal rather than
# from that module because that module pulls in torch-dependent backends.
MIN_LENGTH_CHANGE = 0.10


def read_jsonl(path: str) -> list[dict]:
    rows, bad = [], 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
    if bad:
        logger.warning("%d unparseable lines skipped in %s", bad, path)
    return rows


def relabel(in_path: str, out_path: str, min_length_change: float = MIN_LENGTH_CHANGE,
            keep_failed_augmentation: bool = False) -> dict:
    g2p_manifest = assert_g2p_available()
    rows = read_jsonl(in_path)
    logger.info("Read %d rows from %s", len(rows), in_path)

    by_lang: dict[str, list[int]] = defaultdict(list)
    for i, r in enumerate(rows):
        by_lang[r.get("language", "unknown")].append(i)

    new_n: dict[int, int] = {}
    for lang, idxs in by_lang.items():
        if lang not in LANGUAGES:
            logger.warning("language '%s' is not in the supported table — %d rows left "
                           "untouched and flagged", lang, len(idxs))
            continue
        targets = [(rows[i].get("target") or rows[i].get("completion") or "") for i in idxs]
        counts = [len(toks) for toks in phonemize_many(targets, lang)]
        for i, c in zip(idxs, counts):
            new_n[i] = c
        logger.info("relabelled %-3s: %5d rows", lang, len(idxs))

    # Index base (non-augmented) rows so augmented variants can be re-tested against a
    # source measured in the same unit they now are.
    base_true: dict[tuple, int] = {}
    for i, r in enumerate(rows):
        if "augmentation" not in r and i in new_n:
            base_true[(r.get("language"), r.get("english"))] = new_n[i]

    stats: dict[str, dict] = defaultdict(lambda: {
        "n": 0, "n_augmented": 0, "dropped_augmentation": 0, "unresolved_source": 0,
        "label_before": [], "label_after": [],
    })
    out_rows: list[dict] = []
    dropped = 0

    for i, r in enumerate(rows):
        lang = r.get("language")
        if i not in new_n:
            out_rows.append({**r, "ruler": "unverified", "relabel_skipped": True})
            continue

        n = new_n[i]
        if n <= 0:
            dropped += 1
            continue

        s = stats[lang]
        s["n"] += 1
        s["label_before"].append(int(r.get("n_phonemes") or 0))
        s["label_after"].append(n)

        new_row = dict(r)
        new_row["n_phonemes_legacy"] = r.get("n_phonemes")
        new_row["n_chars"] = count_chars(r.get("target") or r.get("completion") or "")
        new_row["n_phonemes"] = n
        new_row["ruler"] = g2p_manifest["ruler"]
        new_row["prompt"] = PROMPT_TEMPLATE.format(
            language=LANGUAGES[lang].name, n_phonemes=n, english=r.get("english", ""),
        )

        aug = r.get("augmentation")
        if aug:
            s["n_augmented"] += 1
            src = base_true.get((lang, r.get("english")))
            if src is None or src <= 0:
                # No base row in this file to measure against. Keep it, but say so —
                # silently trusting the old mixed-unit number is what got us here.
                s["unresolved_source"] += 1
                new_row["augmentation"] = {
                    **aug,
                    "source_phonemes_legacy": aug.get("source_phonemes"),
                    "source_phonemes": None,
                    "length_change_revalidated": False,
                    "revalidation_note": "base row not present in this file",
                }
            else:
                change = (n - src) / src
                wanted = -1 if aug.get("direction") == "compress" else 1
                passed = (wanted * change) >= min_length_change
                new_row["augmentation"] = {
                    **aug,
                    "source_phonemes_legacy": aug.get("source_phonemes"),
                    "source_phonemes": src,
                    "relative_change": round(change, 4),
                    "length_change_revalidated": True,
                    "length_gate_passed": passed,
                }
                if not passed:
                    s["dropped_augmentation"] += 1
                    if not keep_failed_augmentation:
                        dropped += 1
                        continue

        out_rows.append(new_row)

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    per_lang = {}
    for lang, s in sorted(stats.items()):
        before, after = s["label_before"], s["label_after"]
        changed = sum(1 for b, a in zip(before, after) if b != a)
        per_lang[lang] = {
            "n": s["n"],
            "n_augmented": s["n_augmented"],
            "labels_changed": changed,
            "labels_changed_frac": round(changed / max(s["n"], 1), 4),
            "mean_label_before": round(statistics.fmean(before), 2) if before else None,
            "mean_label_after": round(statistics.fmean(after), 2) if after else None,
            "mean_shift_pct": (
                round(100 * (statistics.fmean(after) / statistics.fmean(before) - 1), 2)
                if before and statistics.fmean(before) else None
            ),
            "augmentation_failed_revalidation": s["dropped_augmentation"],
            "augmentation_source_unresolved": s["unresolved_source"],
        }

    manifest = {
        "tool": "tools/relabel_dataset.py",
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "input": in_path,
        "output": out_path,
        "ruler": g2p_manifest["ruler"],
        "espeak_version": g2p_manifest["espeak_version"],
        "rows_in": len(rows),
        "rows_out": len(out_rows),
        "rows_dropped": dropped,
        "min_length_change": min_length_change,
        "keep_failed_augmentation": keep_failed_augmentation,
        "per_language": per_lang,
    }
    Path(str(out_path) + ".manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return manifest


def main() -> int:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--in", dest="in_path", required=True)
    p.add_argument("--out", dest="out_path", required=True)
    p.add_argument("--min_length_change", type=float, default=MIN_LENGTH_CHANGE)
    p.add_argument("--keep_failed_augmentation", action="store_true",
                   help="Retain augmented rows that fail the re-validated length gate, "
                        "flagged rather than dropped.")
    args = p.parse_args()

    m = relabel(args.in_path, args.out_path, args.min_length_change,
                args.keep_failed_augmentation)

    print(f"\nruler: {m['ruler']}")
    print(f"rows: {m['rows_in']} in -> {m['rows_out']} out ({m['rows_dropped']} dropped)\n")
    hdr = (f"{'lang':<5}{'n':>7}{'aug':>6}{'changed':>9}{'before':>9}{'after':>9}"
           f"{'shift%':>9}{'augFail':>9}")
    print(hdr)
    print("-" * len(hdr))
    for lang, v in m["per_language"].items():
        print(f"{lang:<5}{v['n']:>7}{v['n_augmented']:>6}{v['labels_changed_frac']:>9.2f}"
              f"{v['mean_label_before'] or 0:>9.1f}{v['mean_label_after'] or 0:>9.1f}"
              f"{v['mean_shift_pct'] or 0:>9.1f}{v['augmentation_failed_revalidation']:>9}")
    print(f"\nWrote {m['output']} and {m['output']}.manifest.json")
    print("Next: python -m tools.ruler_audit --jsonl " + m["output"])
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 3. G2P preflight — the check that would have caught this on day one

Asserting that `phonemizer` *imports* is not enough: the failure being recovered from produced plausible, non-empty output. This asserts the returned symbols are **phonemes**, not the input's own characters.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

from common.phonemes import assert_g2p_available, ruler_id
manifest = assert_g2p_available()
print("\nRULER:", ruler_id())


## 4. Locate the real corpus

Searched rather than hard-coded, so the notebook reports what it actually audited instead of silently auditing the wrong file. The fine-tune used 53,350 train / 1,650 val rows — check the counts below match.

In [ ]:
from pathlib import Path

candidates = []
for p in Path("/kaggle/input").rglob("*.jsonl"):
    try:
        n = sum(1 for _ in open(p, encoding="utf-8"))
    except Exception:
        continue
    candidates.append((n, str(p)))
candidates.sort(reverse=True)

print(f"{'rows':>8}  path")
for n, p in candidates[:25]:
    print(f"{n:>8}  {p}")

TRAIN = next((p for n, p in candidates if p.endswith("/train.jsonl")), None)
VAL   = next((p for n, p in candidates if p.endswith("/val.jsonl")), None)
# train_final.jsonl is notebook 03's output: the base corpus PLUS the length-augmented
# rows. It is audited too, because it is where the second ruler lives — the augmentation
# session had espeak-ng and wrote real phoneme counts alongside the base rows' character
# counts. It is also, per the runbook, a candidate for what the 3,801-step run actually
# consumed, and that ambiguity is exactly what an audit should settle rather than assume.
AUG   = next((p for n, p in candidates if p.endswith("/train_final.jsonl")), None)

FILES = [f for f in (TRAIN, VAL, AUG) if f]
print("\nTRAIN:", TRAIN)
print("VAL:  ", VAL)
print("AUG:  ", AUG)
assert TRAIN and VAL, ("could not find train.jsonl / val.jsonl — attach the notebook "
                       "output that contains them (Add Input -> Your Work -> Notebooks)")


## 5. The audit

Per language: which ruler wrote the labels, the phonemes-per-character ratio and its coefficient of variation, and the resulting SALVAGE / RESCALE_THEN_MEASURE / RETRAIN verdict.

`=chars` near 1.00 confirms the character-count identity. `CV` is the number the decision turns on.

In [ ]:
import json
from pathlib import Path
from tools.ruler_audit import audit_file, _print_table, _print_decision

Path("/kaggle/working/audit").mkdir(parents=True, exist_ok=True)

results = [audit_file(f) for f in FILES]
for r in results:
    _print_table(r)
_print_decision(results)

Path("/kaggle/working/audit/ruler_audit_before.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")


## 6. Repair the corpus

Recomputes every label with the canonical counter, rewrites the prompts to match — leaving the prompt stale while fixing the label would be strictly worse than doing nothing — and re-validates the augmentation length gate with both sides measured in the same unit. The original gate divided a phoneme count by a character count and called the result a relative length change; it is not one, and its sign was untrustworthy.

In [ ]:
from tools.relabel_dataset import relabel

Path("/kaggle/working/data").mkdir(parents=True, exist_ok=True)
OUT_FOR = {f: f"/kaggle/working/data/{Path(f).stem}.phonemes.jsonl" for f in FILES}
manifests = [relabel(f, OUT_FOR[f]) for f in FILES]

for m in manifests:
    print(f"\n{m['input']}  ->  {m['output']}")
    print(f"  rows {m['rows_in']} -> {m['rows_out']} ({m['rows_dropped']} dropped)")
    hdr = f"{'lang':<5}{'n':>7}{'aug':>6}{'changed':>9}{'before':>9}{'after':>9}{'shift%':>9}{'augFail':>9}"
    print("  " + hdr)
    for lang, v in m["per_language"].items():
        print(f"  {lang:<5}{v['n']:>7}{v['n_augmented']:>6}{v['labels_changed_frac']:>9.2f}"
              f"{v['mean_label_before'] or 0:>9.1f}{v['mean_label_after'] or 0:>9.1f}"
              f"{v['mean_shift_pct'] or 0:>9.1f}{v['augmentation_failed_revalidation']:>9}")


## 7. Confirm the repair

Must print `PASS: every audited file is phoneme-ruled.` A relabel that is not re-audited is a relabel you cannot trust — that is the whole lesson of this notebook applied to its own output.

In [ ]:
after = [audit_file(OUT_FOR[f]) for f in FILES]
for r in after:
    _print_table(r)

Path("/kaggle/working/audit/ruler_audit_after.json").write_text(
    json.dumps(after, ensure_ascii=False, indent=2), encoding="utf-8")

from common.phonemes import RULER_PHONEMES
bad = [r["file"] for r in after if r["corpus_ruler"] != RULER_PHONEMES]
assert not bad, f"relabel did NOT produce a phoneme-ruled corpus: {bad}"
print("\nPASS: every audited file is phoneme-ruled.")


## 8. Emit the decision

Written to disk so the next session reads a recorded decision rather than re-deriving it, and so any number quoted later traces to a file.

In [ ]:
import json
from pathlib import Path

before = json.loads(Path("/kaggle/working/audit/ruler_audit_before.json").read_text(encoding="utf-8"))
k = {lang: v["ols_k_through_origin"] for lang, v in before[0]["per_language"].items()}
Path("/kaggle/working/audit/budget_scale.json").write_text(json.dumps(k, indent=2), encoding="utf-8")

verdict = before[0]["corpus_verdict"]
NEXT = {
    "SALVAGE":
        "No retraining. Run Session B with --budget_scale_json audit/budget_scale.json; the "
        "relabelled corpus is for the NEXT training run, not this one.",
    "RESCALE_THEN_MEASURE":
        "Run Session B with --budget_scale_json and compare the probe slope against the "
        "unscaled baseline. Retrain only if the residual is still large.",
    "RETRAIN":
        "Retrain from data/train.phonemes.jsonl. Fresh run, not a resume — the old "
        "checkpoints were conditioned on a different unit.",
}
summary = {
    "verdict": verdict,
    "corpus_ruler_before": before[0]["corpus_ruler"],
    "scoring_ruler": before[0]["scoring_ruler"],
    "phonemes_per_char": k,
    "per_language_cv": {lang: v["phonemes_per_char_cv"]
                        for lang, v in before[0]["per_language"].items()},
    "next_step": NEXT[verdict],
}
Path("/kaggle/working/audit/DECISION.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("\n>>> Save Version now. Session B attaches this output for "
      "data/*.phonemes.jsonl and audit/budget_scale.json.")
